# HAI Digital Twin — EDA

Full exploratory data analysis in two parts:

| Part | Focus |
|------|-------|
| **Part 1 — Dataset & Merge Validation** | How HAI + HAIEnd are combined, timestamp alignment, duplicate columns, row counts |
| **Part 2 — Data EDA (Steps 1–10)** | Schema, missing values, label distribution, correlations, sensor distributions, attack timeline |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# Make utils importable from the notebooks/ directory
sys.path.insert(0, str(Path('..').resolve()))

from utils.data_loader import load_merged, HAI_DIR, HIEND_DIR, HAI_DUPLICATES
from utils.schema import META_COLS, select_critical_sensors

---
## Part 1 — Dataset & Merge Validation

The HAI dataset consists of two separate CSV collections covering the same physical system and time windows:

| Dataset | Files | Columns | Description |
|---------|-------|---------|-------------|
| **HAI** | `hai-train1..4.csv`, `hai-test1..2.csv` | 87 | Sensor readings from P1/P2/P3/P4 subsystems |
| **HAIEnd** | `end-train1..4.csv`, `end-test1..2.csv` | 226 | Extended sensors: DM-* signals, PLC outputs |

They are merged on timestamp (inner join). 35 columns appear in both — HAIEnd names take priority.

### 1.1 Load all splits

In [ ]:
train_data = {f'train{i}': load_merged('train', i) for i in range(1, 5)}
test_data  = {f'test{i}':  load_merged('test',  i) for i in range(1, 3)}
all_splits = {**train_data, **test_data}

summary = []
for name, df in all_splits.items():
    summary.append({
        'split':   name,
        'rows':    len(df),
        'columns': len(df.columns),
        'start':   df['timestamp'].iloc[0],
        'end':     df['timestamp'].iloc[-1],
        'attack%': f"{df['attack'].mean()*100:.2f}%" if 'attack' in df.columns else 'N/A',
    })

pd.DataFrame(summary).set_index('split')

### 1.2 Timeline of all splits

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
colors = {'train': '#2ecc71', 'test': '#e74c3c'}

for i, (name, df) in enumerate(all_splits.items()):
    kind  = 'train' if 'train' in name else 'test'
    start = df['timestamp'].iloc[0]
    end   = df['timestamp'].iloc[-1]
    ax.barh(i, (end - start).total_seconds() / 86400,
            left=mdates.date2num(start),
            height=0.5, color=colors[kind], alpha=0.85)
    ax.text(mdates.date2num(start), i,
            f'  {name}  ({len(df):,} rows)', va='center', fontsize=9)

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.set_xlabel('Date')
ax.set_yticks([])
ax.set_title('Timeline of merged splits  (green = train, red = test)')
plt.tight_layout()
plt.show()

### 1.3 Column composition after deduplication

35 HAI columns have authoritative HAIEnd counterparts and are dropped before merging. After deduplication the merged dataset has **277 sensor columns + timestamp**.

In [ ]:
df0 = train_data['train1']
hai_only_cols   = [c for c in df0.columns
                   if c.startswith(('P1_','P2_','P3_','P4_','x')) and c != 'timestamp']
hiend_only_cols = [c for c in df0.columns
                   if not c.startswith(('P1_','P2_','P3_','P4_','x')) and c != 'timestamp']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie([len(hai_only_cols), len(hiend_only_cols)],
            labels=['HAI-only cols', 'HAIEnd cols'],
            colors=['#3498db', '#e67e22'], autopct='%1.0f%%',
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title(f'Column composition (total = {len(hai_only_cols)+len(hiend_only_cols)} + timestamp)')

bars = axes[1].bar(['HAI raw', 'HAIEnd raw', 'Merged (deduped)'],
                   [87, 226, len(df0.columns)],
                   color=['#3498db', '#e67e22', '#2ecc71'], width=0.5)
axes[1].set_ylabel('Number of columns')
axes[1].set_title('Column count: raw vs merged')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1, str(int(bar.get_height())),
                 ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f'HAI_DUPLICATES dropped ({len(HAI_DUPLICATES)} cols):')
print(', '.join(HAI_DUPLICATES))

### 1.4 Duplicate sensor pair validation

HAI and HAIEnd record the same physical signals under different names. The plots below confirm the readings are identical — so dropping the HAI version is safe.

In [ ]:
import os

hai_raw   = pd.read_csv(os.path.join(HAI_DIR,   'hai-train1.csv'),   nrows=500)
hiend_raw = pd.read_csv(os.path.join(HIEND_DIR, 'end-train1.csv'), nrows=500)
hai_raw.rename(columns={hai_raw.columns[0]: 'timestamp'}, inplace=True)
hiend_raw.rename(columns={hiend_raw.columns[0]: 'timestamp'}, inplace=True)
hai_raw['timestamp']   = pd.to_datetime(hai_raw['timestamp'])
hiend_raw['timestamp'] = pd.to_datetime(hiend_raw['timestamp'])
raw_merged = pd.merge(hai_raw, hiend_raw, on='timestamp', how='inner', suffixes=('', '_hiend'))

pairs = [
    ('P1_FT01',  'DM-FT01',  'Flow rate FT01'),
    ('P1_LIT01', 'DM-LIT01', 'Water level LIT01'),
    ('P1_TIT01', 'DM-TIT01', 'Temperature TIT01'),
    ('P1_PIT01', 'DM-PIT01', 'Pressure PIT01'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for ax, (hai_col, hiend_col, title) in zip(axes.flat, pairs):
    ax.plot(raw_merged['timestamp'], raw_merged[hai_col],
            label=f'HAI: {hai_col}', color='#3498db', linewidth=1.2)
    ax.plot(raw_merged['timestamp'], raw_merged[hiend_col],
            label=f'HAIEnd: {hiend_col}', color='#e74c3c', linewidth=1, linestyle='--')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

fig.suptitle('HAI vs HAIEnd readings for duplicate pairs (train1, first 500 rows)', fontsize=12)
plt.tight_layout()
plt.show()

### 1.5 Row counts after inner join

HAIEnd starts 1 second earlier than HAI — a consistent recording lag. The inner join on exact timestamp drops only that 1 boundary row per file. No values are manipulated or shifted.

In [ ]:
raw_map = {
    'train1': 280800, 'train2': 291600, 'train3': 126000, 'train4': 198000,
    'test1':   54000, 'test2':  230400,
}
splits        = list(all_splits.keys())
raw_counts    = [raw_map[s] for s in splits]
merged_counts = [len(all_splits[s]) for s in splits]
dropped       = [r - m for r, m in zip(raw_counts, merged_counts)]

x = range(len(splits))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, raw_counts,    color='#bdc3c7', label='Raw HAI rows', width=0.5)
ax.bar(x, merged_counts, color='#2ecc71', label='Kept after inner join', width=0.5)
ax.set_xticks(list(x))
ax.set_xticklabels(splits)
ax.set_ylabel('Row count')
ax.set_title('Rows kept vs dropped by inner join (1 row dropped per file)')
ax.legend()
for i, d in enumerate(dropped):
    ax.text(i, raw_counts[i] + 500, f'-{d}', ha='center', fontsize=9, color='#e74c3c')
plt.tight_layout()
plt.show()

---
## Part 2 — Data EDA (Steps 1–10)

All steps run on **test2** — the held-out evaluation split. It contains both normal and attack rows, making it the most informative split for EDA.

In [ ]:
df = test_data['test2']
sensor_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in META_COLS]
print(f'test2: {df.shape[0]:,} rows x {df.shape[1]} cols  |  {len(sensor_cols)} sensor cols')

### Step 1 — Schema Summary

In [ ]:
n_rows, n_cols = df.shape
memory_mb      = df.memory_usage(deep=True).sum() / 1024**2
vc             = df['attack'].value_counts().sort_index()
attack_rate    = vc.get(1, 0) / n_rows * 100
ts             = df['timestamp']
duration_h     = (ts.max() - ts.min()).total_seconds() / 3600

print(f'Shape:          {n_rows:,} rows x {n_cols} cols')
print(f'Memory:         {memory_mb:.1f} MB')
print(f'Time range:     {ts.min()}  →  {ts.max()}')
print(f'Duration:       {duration_h:.1f} hours')
print(f'Attack rate:    {attack_rate:.3f}%')
print(f'Label counts:   {vc.to_dict()}')
print(f'Total nulls:    {df.isnull().sum().sum():,}')
print(f'Duplicate rows: {df.duplicated().sum():,}')

### Step 2 — Missing Values

In [ ]:
null_pct = df.isnull().mean() * 100
top_cols = null_pct[null_pct > 0].nlargest(50)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Missing Values Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
if len(top_cols) > 0:
    top_cols.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Top {len(top_cols)} Columns by Missing %')
    ax.set_ylabel('Missing %')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.axhline(y=5, color='red', linestyle='--', linewidth=0.8, alpha=0.7, label='5% threshold')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'No missing values found', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    ax.set_title('Missing Values — Bar Chart')

ax2 = axes[1]
sample = df.sample(min(1000, len(df)), random_state=42)
cols_with_nulls = sample.columns[sample.isnull().any()].tolist()[:50]
if cols_with_nulls:
    missing_matrix = sample[cols_with_nulls].isnull().astype(int)
    im = ax2.imshow(missing_matrix.T, aspect='auto', cmap='RdYlGn_r',
                    interpolation='none', vmin=0, vmax=1)
    ax2.set_yticks(range(len(cols_with_nulls)))
    ax2.set_yticklabels(cols_with_nulls, fontsize=6)
    ax2.set_xlabel(f'Sample rows (n={len(sample)})')
    ax2.set_title('Missing Pattern Heatmap')
    plt.colorbar(im, ax=ax2, label='Missing (1=yes)')
else:
    ax2.text(0.5, 0.5, 'No missing values in sample', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Missing Pattern Heatmap')

plt.tight_layout()
plt.show()

### Step 3 — Label Distribution

In [ ]:
vc      = df['attack'].value_counts().sort_index()
labels  = [('Normal' if k == 0 else 'Attack') for k in vc.index]
counts  = vc.values
pcts    = counts / counts.sum() * 100
colors  = ['steelblue', 'tomato']
ratio   = counts[0] / counts[1] if len(counts) > 1 and counts[1] > 0 else float('inf')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Label Distribution', fontsize=14, fontweight='bold')

bars = axes[0].bar(labels, counts, color=colors[:len(labels)], edgecolor='white', width=0.5)
for bar, pct, cnt in zip(bars, pcts, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + counts.max()*0.01,
                 f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_title(f'Class Counts  |  Imbalance ratio {ratio:.1f}:1')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, counts.max() * 1.15)
axes[0].grid(axis='y', alpha=0.3)

axes[1].pie(counts, labels=labels, colors=colors[:len(labels)],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title(f'Attack Rate: {pcts[1]:.2f}%' if len(pcts) > 1 else 'All Normal')

plt.tight_layout()
plt.show()

### Step 4 — Descriptive Statistics

In [ ]:
top50_cols = sensor_cols[:50]
desc = df[top50_cols].describe().T
desc['cv'] = desc['std'] / (desc['mean'].abs() + 1e-9)

print('Top 5 by std:')
print(desc['std'].nlargest(5).to_string())
print('\nTop 5 by coefficient of variation (cv = std/mean):')
print(desc['cv'].nlargest(5).to_string())
desc.head(10)

### Step 5 — Outlier Detection (IQR × 3)

In [ ]:
outlier_counts = {}
for col in top50_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = int(((df[col] < q1 - 3*iqr) | (df[col] > q3 + 3*iqr)).sum())
    if n_out > 0:
        outlier_counts[col] = n_out

print(f'Columns with outliers: {len(outlier_counts)} / {len(top50_cols)}')
print('\nTop 10 offenders:')
for col, cnt in sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {col:<35s}  {cnt:6,}  ({cnt/len(df)*100:.2f}%)')

### Step 6 — Correlation Heatmap

In [ ]:
top40 = df[sensor_cols].var().nlargest(40).index.tolist()
corr  = df[top40].corr()
mask  = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(np.where(mask, np.nan, corr.values),
               cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson r', shrink=0.8)
ax.set_xticks(range(len(top40)))
ax.set_yticks(range(len(top40)))
ax.set_xticklabels(top40, rotation=90, fontsize=6)
ax.set_yticklabels(top40, fontsize=6)
ax.set_title('Correlation Heatmap — Top 40 sensors by variance (lower triangle, Pearson r)', fontsize=11)
plt.tight_layout()
plt.show()

atk_corr = df[top40 + ['attack']].corr()['attack'].drop('attack').abs()
print('\nTop 10 features by |correlation| with attack label:')
for feat, val in atk_corr.nlargest(10).items():
    print(f'  {feat:<35s}  {val:.4f}')

### Step 7 — Sensor Distributions (Normal vs Attack)

In [ ]:
top20   = df[sensor_cols].var().nlargest(20).index.tolist()
normal  = df[df['attack'] == 0]
attack  = df[df['attack'] == 1]
n_cols  = 4
n_rows_fig = (len(top20) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows_fig, n_cols, figsize=(5*n_cols, 3.5*n_rows_fig))
axes = np.array(axes).flatten()
fig.suptitle('Sensor Distribution — Normal vs Attack', fontsize=13, fontweight='bold')

for i, col in enumerate(top20):
    ax = axes[i]
    ax.hist(normal[col].dropna(), bins=50, alpha=0.6, color='steelblue',
            label='Normal', density=True)
    ax.hist(attack[col].dropna(), bins=50, alpha=0.6, color='tomato',
            label='Attack', density=True)
    ax.set_title(col, fontsize=8)
    ax.tick_params(labelsize=6)
    ax.set_ylabel('Density', fontsize=6)
    if i == 0:
        ax.legend(fontsize=7)

for j in range(len(top20), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### Step 8 — Attack Timeline

In [ ]:
critical = select_critical_sensors(df, n=10)
print('Critical sensors (top variance + |corr with attack|):')
for s in critical:
    print(f'  {s}')

sensor = critical[0]
x = pd.to_datetime(df['timestamp'])

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
fig.suptitle('Attack Timeline', fontsize=13, fontweight='bold')

axes[0].fill_between(x, df['attack'], color='tomato', alpha=0.7, step='post')
axes[0].set_ylabel('Attack Label')
axes[0].set_ylim(-0.05, 1.3)
axes[0].set_yticks([0, 1])
axes[0].set_title('Attack Label (1 = Attack)')
axes[0].grid(axis='y', alpha=0.3)

attack_mask = df['attack'].values.astype(bool)
axes[1].plot(x, df[sensor], color='steelblue', linewidth=0.5, label=sensor)
axes[1].fill_between(x, df[sensor].min(), df[sensor].max(),
                     where=attack_mask, color='tomato', alpha=0.25, label='Attack window')
axes[1].set_ylabel(sensor)
axes[1].set_title(f'Sensor: {sensor}')
axes[1].legend(fontsize=8)
axes[1].set_xlabel('Time')

plt.tight_layout()
plt.show()

### Step 9 — Rolling Statistics (window = 300s)

In [ ]:
window     = 300
sensors    = [s for s in critical if s in df.columns][:4]
sub        = df.iloc[:10_000].copy()
n          = len(sensors)

fig, axes = plt.subplots(n, 2, figsize=(14, 3.5*n))
if n == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'Rolling Statistics (window={window}s, first 10,000 rows)', fontsize=12, fontweight='bold')

for i, sensor in enumerate(sensors):
    series    = sub[sensor].ffill()
    roll_mean = series.rolling(window, min_periods=1).mean()
    roll_std  = series.rolling(window, min_periods=1).std().fillna(0)

    axes[i, 0].plot(series.values, color='lightgray', linewidth=0.4, label='raw')
    axes[i, 0].plot(roll_mean.values, color='steelblue', linewidth=1.2, label=f'mean(w={window})')
    axes[i, 0].set_title(f'{sensor} — Rolling Mean', fontsize=9)
    axes[i, 0].set_ylabel('Value', fontsize=8)
    axes[i, 0].legend(fontsize=7)
    axes[i, 0].grid(alpha=0.3)

    axes[i, 1].plot(roll_std.values, color='coral', linewidth=1.0, label=f'std(w={window})')
    axes[i, 1].set_title(f'{sensor} — Rolling Std', fontsize=9)
    axes[i, 1].set_ylabel('Std', fontsize=8)
    axes[i, 1].legend(fontsize=7)
    axes[i, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Step 10 — Attack Segment Analysis

In [ ]:
labels_arr = df['attack'].values
segments, in_seg, start = [], False, 0

for i, v in enumerate(labels_arr):
    if v == 1 and not in_seg:
        in_seg, start = True, i
    elif v == 0 and in_seg:
        in_seg = False
        segments.append({'start': start, 'end': i-1, 'duration': i - start})
if in_seg:
    segments.append({'start': start, 'end': len(labels_arr)-1,
                     'duration': len(labels_arr) - start})

durations = [s['duration'] for s in segments]
print(f'Total attack segments: {len(segments)}')
print(f'Min duration:          {min(durations)}s  ({min(durations)//60}:{min(durations)%60:02d} mm:ss)')
print(f'Max duration:          {max(durations)}s  ({max(durations)//60}:{max(durations)%60:02d} mm:ss)')
mean_d = int(np.mean(durations))
print(f'Mean duration:         {mean_d}s  ({mean_d//60}:{mean_d%60:02d} mm:ss)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(durations)), sorted(durations, reverse=True), color='tomato', alpha=0.8)
ax.axhline(mean_d, color='black', linestyle='--', linewidth=1, label=f'mean = {mean_d}s')
ax.set_xlabel('Segment (sorted by duration)')
ax.set_ylabel('Duration (seconds)')
ax.set_title('Attack Segment Durations')
ax.legend()
plt.tight_layout()
plt.show()